# Installing C extensions at runtime

This notebook demonstrates conda-express's ability to **install packages with
native C extensions at runtime** in the browser. The installed shared libraries
(`.so` files) are automatically loaded into the WASM runtime so Python can
`import` them immediately.

We'll install [lz4](https://github.com/python-lz4/python-lz4) — a Python
binding for the [LZ4 compression algorithm](https://lz4.org/) — which includes
both a C library (`liblz4`) and Python C extension modules.

*The lz4 compression examples below are adapted from the
[python-lz4 documentation](https://python-lz4.readthedocs.io/) (BSD 3-Clause license).*

## Step 1: Install lz4 with conda

This installs the `lz4` Python package and its C dependency `lz4-c` from
[emscripten-forge](https://emscripten-forge.org/). After installation, shared
libraries like `liblz4.so` are automatically loaded into the WASM runtime.

In [ ]:
%load_ext conda_emscripten

In [ ]:
%conda install lz4

## Step 2: Verify the C extension loaded

The `lz4` package contains C extension modules (`.so` files compiled to WASM)
that call into the native `liblz4` C library. If this import works, the shared
library loading chain is functioning correctly.

In [ ]:
import lz4
import lz4.frame
import lz4.block

print(f"lz4 version:   {lz4.VERSION}")
print(f"lz4-c version: {lz4.library_version_string()}")
print(f"Modules:       lz4.frame, lz4.block (C extensions)")

## Step 3: Block compression

`lz4.block` provides low-level compression for individual data blocks.
LZ4 is designed for speed — it's one of the fastest compression algorithms
available.

In [ ]:
import numpy as np

original = np.random.bytes(100_000)
compressed = lz4.block.compress(original)
decompressed = lz4.block.decompress(compressed, uncompressed_size=len(original))

ratio = len(compressed) / len(original) * 100
print(f"Original:     {len(original):>8,} bytes")
print(f"Compressed:   {len(compressed):>8,} bytes ({ratio:.1f}%)")
print(f"Round-trip OK: {decompressed == original}")

## Step 4: Frame compression

`lz4.frame` provides higher-level framed compression, suitable for streaming
and file-like usage. Let's compress some structured text data where LZ4 really
shines.

In [ ]:
import json

data = json.dumps([
    {"id": i, "name": f"package-{i}", "version": f"1.{i}.0", "build": "h1234_0"}
    for i in range(1000)
]).encode()

compressed = lz4.frame.compress(data)
decompressed = lz4.frame.decompress(compressed)

ratio = len(compressed) / len(data) * 100
print(f"JSON data:    {len(data):>8,} bytes")
print(f"Compressed:   {len(compressed):>8,} bytes ({ratio:.1f}%)")
print(f"Round-trip OK: {decompressed == data}")
print(f"\nThis is the same C code that runs natively on your laptop —")
print(f"here it's compiled to WebAssembly and loaded at runtime.")

## How shared library loading works

When `%conda install lz4` completes, conda-express automatically:

1. **Scans** for new `.so` files in the prefix (e.g. `liblz4.so.1.10.0`,
   `_frame.cpython-313-wasm32-emscripten.so`)
2. **Sorts** by directory depth (C runtime libraries before Python extension modules)
3. **Loads** each via `ctypes.CDLL` with `RTLD_GLOBAL`, which calls Emscripten's
   `dlopen` → `loadDynamicLibrary`
4. **Retries** once on failure (handles dependency ordering between libraries)

This enables any package with C extensions to work after runtime installation,
as long as the package is available on
[emscripten-forge](https://emscripten-forge.org/).